In [ ]:
from IPython.display import Image, display

In [ ]:
import numpy as np
import polars as pl
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.GraphUtils import GraphUtils

RANDOM_SEED = 202605211523

In [ ]:
from climate_attitudes.dataset import Dataset
from climate_attitudes.datasets import reduced_no_imputation as ds_spec
from climate_attitudes.settings import Config

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="reduced_no_imputation", with_imputation=False)
indices = (
    dataset.indices.collect()  # ty: ignore
)

In [ ]:
labels = [ds_spec.RENAME[col] for col in indices.columns[6:]]

In [ ]:
n_participants = indices.select(pl.col("participant_id").unique()).shape[0]
n_waves = indices.select(pl.col("wave").unique()).shape[0]

# Extract indices into matrix (particiant, wave, index)
data = (
    indices.drop("dem_male", "dem_educ", "dem_income_percep", "dem_urban")
    .sort(by=("participant_id", "wave"))
    .drop("participant_id", "wave")
    .to_numpy()
    .ravel()
    .reshape((n_participants, n_waves, -1))
)

# Binarise into (-1, +1) with added noise to smooth binarisation
rng = np.random.default_rng(RANDOM_SEED)
ζ = rng.normal(scale=0.1, size=data.size).reshape(data.shape)
data = data + ζ
data = np.where(data < 0.0, -1, 1).astype(np.int64)

## 1. CD with cross-sectional data

In [ ]:
cd_data_1 = data.swapaxes(0, 1).ravel().reshape(n_participants * n_waves, -1)
g1, edges1 = fci(cd_data_1)

In [ ]:
pdy1 = GraphUtils.to_pydot(g1, labels=labels)
png1 = pdy1.create_png()
display(Image(png1))

## 2. CD with temporal info

1. Require self-interactions $s_i^t \to s_i^{t + dt}$
2. Disallow instantaneous interactions $s_i^t \to s_j^t$ for $i != j$
3. Disallow reverse-time interactions $s_i^{t + dt} \to s_j^t$

In [ ]:
from causallearn.graph.GraphClass import CausalGraph
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge

In [ ]:
cd_data_2 = data.ravel().reshape((n_participants, -1))

Add $T=t$ nodes to tier 1, $T=t+dt$ nodes to tier 2. Disallows instantaneous interactions (within a tier) and time-reversed interactions (from $t+dt$ to $t$).

In [ ]:
cg = CausalGraph(cd_data_2.shape[-1])
cg_nodes = cg.G.get_nodes()

bk = BackgroundKnowledge()

for i in range(data.shape[-1]):
    t1_node = cg_nodes[i]
    t2_node = cg_nodes[data.shape[-1] + i]
    bk = bk.add_node_to_tier(t1_node, 1)
    bk = bk.add_node_to_tier(t2_node, 2)

bk.forbid_within_tier(1)
bk.forbid_within_tier(2);

Require self-interaction edges across timesteps

In [ ]:
for i in range(data.shape[-1]):
    t1_node = cg_nodes[i]
    t2_node = cg_nodes[data.shape[-1] + i]
    bk.add_required_by_node(t1_node, t2_node)

In [ ]:
# Check all forbidden:
# 1. Self-interaction within single tier
assert bk.is_forbidden(cg_nodes[0], cg_nodes[0])
assert bk.is_forbidden(cg_nodes[10], cg_nodes[10])

# 2. Non-self interaction within single tier
assert bk.is_forbidden(cg_nodes[0], cg_nodes[1])
assert bk.is_forbidden(cg_nodes[10], cg_nodes[9])

# 3. Time-reversed interactions
assert bk.is_forbidden(cg_nodes[8], cg_nodes[7])
assert bk.is_forbidden(cg_nodes[10], cg_nodes[1])

# Check not forbidden: non-self interactions to next tier
assert not bk.is_forbidden(cg_nodes[0], cg_nodes[9])
assert not bk.is_forbidden(cg_nodes[1], cg_nodes[9])

# Check not required: non-self interactions to next tier
assert not bk.is_required(cg_nodes[0], cg_nodes[10])
assert not bk.is_required(cg_nodes[1], cg_nodes[10])

# Check required: self-interactions to next tier
assert bk.is_required(cg_nodes[0], cg_nodes[8])
assert bk.is_required(cg_nodes[1], cg_nodes[9])
assert bk.is_required(cg_nodes[7], cg_nodes[15])

In [ ]:
g2, edges2 = fci(cd_data_2, background_knowledge=bk)

In [ ]:
ts_labels = [
    f"{label} {suffix}" for suffix in (" (t)", " (t + dt)") for label in labels
]
pdy2 = GraphUtils.to_pydot(g2, labels=ts_labels)
png2 = pdy2.create_png()
display(Image(png2))

In [ ]:
labels

In [ ]:
from causallearn.graph.Edge import Edge
from causallearn.graph.Endpoint import Endpoint
from causallearn.graph.GeneralGraph import GeneralGraph

In [ ]:
nodes = g1.get_nodes()
gg = GeneralGraph(nodes)


def create_edge(graph, i, j) -> Edge:
    n = data.shape[-1]

    endpoints = []
    for ep in (graph[j + n, i], graph[i + n, j]):
        match ep:
            case -1:
                endpoints.append(Endpoint.TAIL)
            case 0:
                endpoints.append(Endpoint.NULL)
            case 1:
                endpoints.append(Endpoint.ARROW)
            case 2:
                endpoints.append(Endpoint.CIRCLE)
            case _:
                raise ValueError(f"Unexpected endpoint type: {ep}")

    has_non_null = False
    for ep in endpoints:
        if ep != Endpoint.NULL:
            has_non_null = True

    if has_non_null:
        for k in range(2):
            if endpoints[k] == Endpoint.NULL:
                endpoints[k] = Endpoint.TAIL

    return Edge(nodes[i], nodes[j], *endpoints)


for i in range(data.shape[-1] - 1):
    for j in range(i + 1, data.shape[-1]):
        gg.add_edge(create_edge(g2.graph, i, j))

In [ ]:
adj = gg.graph.T.copy()
adj[adj != 1] = 0
adj

In [ ]:
pdy3 = GraphUtils.to_pydot(gg, labels=labels)
png3 = pdy3.create_png()
display(Image(png3))

In [ ]:
pdy1 = GraphUtils.to_pydot(g1, labels=labels)
png1 = pdy1.create_png()
display(Image(png1))